# 03 — Silver para Gold

**Arquivo:** 03-silver-para-gold.ipynb
**Objetivo:** agregar a Silver em tabelas analíticas de vacância e evolução do segmento
**Entrada:** dados/silver/imoveis/ (Parquet particionado por ano)
**Saída:** dados/gold/*.parquet
**Onde roda:** local (.venv, DuckDB) — **Camada:** Silver → Gold

In [1]:
# duckdb executa SQL sobre arquivos Parquet sem precisar de servidor de banco.
import duckdb
import pandas as pd
from pathlib import Path

# Caminhos centralizados. Na migração para a nuvem, só estas linhas mudam.
RAIZ = Path("..").resolve()
SILVER = RAIZ / "dados" / "silver"
GOLD = RAIZ / "dados" / "gold"

# Abre um banco DuckDB em memória: existe só enquanto o notebook estiver aberto.
con = duckdb.connect()

# Caminho de todos os Parquet da Silver, em qualquer subpasta de ano.
# .as_posix() troca as barras do Windows (\) por barras normais (/),
# que o DuckDB e os serviços de nuvem entendem sem ambiguidade.
caminho_silver = (SILVER / "imoveis").as_posix() + "/**/*.parquet"

# Cria a view. hive_partitioning = true reconstrói a coluna "ano"
# a partir do nome das pastas ano=2020, ano=2021 etc.
con.execute(f"""
    CREATE OR REPLACE VIEW silver_imoveis AS
    SELECT * FROM read_parquet('{caminho_silver}', hive_partitioning = true)
""")

# Conferência: a view precisa enxergar os mesmos 5.607 registros gravados.
total = con.sql("SELECT COUNT(*) FROM silver_imoveis").fetchone()[0]
print("Registros visíveis na view:", total)
print("Bate com a Silver (5607)?", total == 5607)

Registros visíveis na view: 5607
Bate com a Silver (5607)? True


In [2]:
# Para cada fundo em cada trimestre, conta quantas datas de referência distintas existem.
# HAVING filtra os grupos depois do agrupamento: mantém só os que têm mais de uma data.
# MIN e MAX mostram a primeira e a última data encontradas no grupo.
verificacao = """
    SELECT
        CNPJ_Fundo_Classe,
        ano,
        Trimestre,
        COUNT(DISTINCT Data_Referencia) AS qtd_datas,
        MIN(Data_Referencia)            AS primeira_data,
        MAX(Data_Referencia)            AS ultima_data,
        COUNT(*)                        AS qtd_registros
    FROM silver_imoveis
    GROUP BY CNPJ_Fundo_Classe, ano, Trimestre
    HAVING COUNT(DISTINCT Data_Referencia) > 1
"""

df_conflito = con.sql(verificacao).df()
print("Fundos com mais de uma data no mesmo trimestre:", len(df_conflito))
df_conflito

Fundos com mais de uma data no mesmo trimestre: 1


,CNPJ_Fundo_Classe,ano,Trimestre,qtd_datas,primeira_data,ultima_data,qtd_registros
0,13.012.312/0001-67,2023,1,2,2023-01-31,2023-03-31,2


In [3]:
# Gold 1 — vacância do segmento logístico por trimestre (versão corrigida).
# Substitui a Gold v1, que agrupava por Data_Referencia exata e partia
# trimestres (35 linhas em vez de 26), porque os fundos usam datas de
# fechamento diferentes (último dia do mês × último dia útil).

consulta = """
    -- Etapa 1: para cada fundo em cada trimestre, identifica a data de
    -- fechamento (a mais recente). Serve de "régua" para o filtro seguinte.
    WITH ultima_data AS (
        SELECT CNPJ_Fundo_Classe, ano, Trimestre,
               MAX(Data_Referencia) AS data_max
        FROM silver_imoveis
        GROUP BY CNPJ_Fundo_Classe, ano, Trimestre
    ),

    -- Etapa 2: mantém só os registros cuja data é a de fechamento do seu
    -- fundo no trimestre. O JOIN exige fundo, ano, trimestre e data iguais;
    -- faltar uma dessas chaves faria linhas se multiplicarem no cruzamento.
    imoveis_trimestre AS (
        SELECT s.*
        FROM silver_imoveis AS s
        JOIN ultima_data AS u
          ON  s.CNPJ_Fundo_Classe = u.CNPJ_Fundo_Classe
          AND s.ano              = u.ano
          AND s.Trimestre        = u.Trimestre
          AND s.Data_Referencia  = u.data_max
    )

    -- Etapa 3: agrega o segmento por trimestre. A vacância ponderada divide
    -- soma de área vaga por soma de área (galpão grande pesa mais); a média
    -- simples fica ao lado só para comparação na análise exploratória.
    SELECT ano, Trimestre,
        CAST(ano AS VARCHAR) || '-T' || CAST(Trimestre AS VARCHAR) AS periodo,
        COUNT(DISTINCT CNPJ_Fundo_Classe) AS qtd_fundos,
        COUNT(*)                          AS qtd_registros,
        SUM(Area)                         AS area_total_m2,
        SUM(Area_Vaga_m2)                 AS area_vaga_m2,
        SUM(Area_Vaga_m2) / SUM(Area)     AS vacancia_ponderada,
        AVG(Percentual_Vacancia)          AS vacancia_media_simples
    FROM imoveis_trimestre
    GROUP BY ano, Trimestre
    ORDER BY ano, Trimestre
"""

# Executa no DuckDB e devolve um DataFrame pandas, formato usado pelo
# to_parquet e, depois, pelos gráficos do notebook 04.
df_gold_segmento = con.sql(consulta).df()

# Conferência de sanidade: 26 trimestres (2020-T1 a 2026-T2) e 5.606
# registros, ou seja, os 5.607 da Silver menos o registro de 31/01/2023
# do fundo com duas datas no mesmo trimestre.
total_trimestres = len(df_gold_segmento)
total_registros = df_gold_segmento['qtd_registros'].sum()
print("Trimestres na Gold:", total_trimestres, "| Esperado 26?", total_trimestres == 26)
print("Registros usados:", total_registros, "| Esperado 5606?", total_registros == 5606)

# Grava a Gold. mkdir com exist_ok não dá erro se a pasta já existe, e o
# to_parquet sobrescreve o arquivo da v1: rodar de novo gera o mesmo resultado.
GOLD.mkdir(parents=True, exist_ok=True)
df_gold_segmento.to_parquet(GOLD / "vacancia_segmento_trimestral.parquet", index=False)

df_gold_segmento

Trimestres na Gold: 26 | Esperado 26? True
Registros usados: 5606 | Esperado 5606? True


,ano,Trimestre,periodo,qtd_fundos,qtd_registros,area_total_m2,area_vaga_m2,vacancia_ponderada,vacancia_media_simples
0,2020,1,2020-T1,23,99,4.787511e+06,233123.349824,0.048694,0.068007
1,2020,2,2020-T2,25,108,4.681332e+06,196819.186400,0.042043,0.063508
2,2020,3,2020-T3,25,114,5.078611e+06,150754.873557,0.029684,0.051050
3,2020,4,2020-T4,24,124,5.561860e+06,274071.186489,0.049277,0.073195
4,2021,1,2021-T1,26,137,7.060462e+06,296456.054023,0.041988,0.064903
5,2021,2,2021-T2,29,145,6.610305e+06,349096.050475,0.052811,0.067418
6,2021,3,2021-T3,29,149,7.213286e+06,365445.889921,0.050663,0.057300
7,2021,4,2021-T4,32,158,7.788187e+06,370654.374789,0.047592,0.053201
8,2022,1,2022-T1,31,225,7.910103e+06,275957.708632,0.034887,0.033396
9,2022,2,2022-T2,34,240,8.254590e+06,180165.018050,0.021826,0.031074


In [4]:
# Inspeção do único conflito de datas: confere se os dois registros do fundo
# 13.012.312/0001-67 no 2023 T1 são o mesmo imóvel (reenvio do informe) ou
# imóveis diferentes (caso em que o descarte vira limitação na DA-24).
inspecao = """
    SELECT CNPJ_Fundo_Classe, Data_Referencia, Nome_Imovel,
           Area, Percentual_Vacancia, Versao
    FROM silver_imoveis
    WHERE CNPJ_Fundo_Classe = '13.012.312/0001-67'
      AND ano = 2023 AND Trimestre = 1
    ORDER BY Data_Referencia
"""
con.sql(inspecao).df()

,CNPJ_Fundo_Classe,Data_Referencia,Nome_Imovel,Area,Percentual_Vacancia,Versao
0,13.012.312/0001-67,2023-01-31,Pilkington,2230.0,0.0,1
1,13.012.312/0001-67,2023-03-31,Pilkington,2230.0,0.0,1


In [5]:
# View com a regra da DA-24 centralizada: cada fundo entra em cada trimestre
# apenas com os registros da data de referência mais recente. As Gold 2 e 3
# leem desta view, para que a regra exista em um único lugar do pipeline.
con.execute("""
    CREATE OR REPLACE VIEW imoveis_trimestre AS
    WITH ultima_data AS (
        SELECT CNPJ_Fundo_Classe, ano, Trimestre,
               MAX(Data_Referencia) AS data_max
        FROM silver_imoveis
        GROUP BY CNPJ_Fundo_Classe, ano, Trimestre
    )
    SELECT s.*,
           CAST(s.ano AS VARCHAR) || '-T' || CAST(s.Trimestre AS VARCHAR) AS periodo
    FROM silver_imoveis AS s
    JOIN ultima_data AS u
      ON  s.CNPJ_Fundo_Classe = u.CNPJ_Fundo_Classe
      AND s.ano              = u.ano
      AND s.Trimestre        = u.Trimestre
      AND s.Data_Referencia  = u.data_max
""")

# Conferência: a view precisa ter os mesmos 5.606 registros usados na Gold 1.
# Se bater, a lógica da view é idêntica à da Gold 1.
total_view = con.sql("SELECT COUNT(*) FROM imoveis_trimestre").fetchone()[0]
print("Registros na view:", total_view, "| Esperado 5606?", total_view == 5606)

Registros na view: 5606 | Esperado 5606? True


In [6]:
# Gold 2 — vacância por fundo por trimestre. Mesma conta da Gold 1 (soma da
# área vaga ÷ soma da área), mas agrupando também por CNPJ. Não há risco de
# divisão por zero: imóveis com área zero foram para a quarentena na Silver.
consulta_fundo = """
    SELECT CNPJ_Fundo_Classe, ano, Trimestre, periodo,
        COUNT(*)                      AS qtd_registros,
        SUM(Area)                     AS area_total_m2,
        SUM(Area_Vaga_m2)             AS area_vaga_m2,
        SUM(Area_Vaga_m2) / SUM(Area) AS vacancia_ponderada
    FROM imoveis_trimestre
    GROUP BY CNPJ_Fundo_Classe, ano, Trimestre, periodo
    ORDER BY ano, Trimestre, CNPJ_Fundo_Classe
"""
df_gold_fundo = con.sql(consulta_fundo).df()

# Conferências cruzadas com a Gold 1. Cada fundo vira uma linha por trimestre,
# então o total de linhas deve ser igual à soma de qtd_fundos da Gold 1. As
# somas de registros e de área vaga devem bater, porque é o mesmo dado cortado
# de outro jeito. A área é comparada por diferença (abs) por causa de
# arredondamentos minúsculos em números decimais.
linhas_esperadas = df_gold_segmento['qtd_fundos'].sum()
diferenca_area = abs(df_gold_fundo['area_vaga_m2'].sum() - df_gold_segmento['area_vaga_m2'].sum())
print("Linhas na Gold 2:", len(df_gold_fundo), "| Bate com a Gold 1?", len(df_gold_fundo) == linhas_esperadas)
print("Registros:", df_gold_fundo['qtd_registros'].sum(), "| Esperado 5606?", df_gold_fundo['qtd_registros'].sum() == 5606)
print("Área vaga bate com a Gold 1?", diferenca_area < 0.01)

# Grava no mesmo padrão da Gold 1: arquivo único, sobrescrito a cada execução.
df_gold_fundo.to_parquet(GOLD / "vacancia_fundo_trimestral.parquet", index=False)
df_gold_fundo.head(10)

Linhas na Gold 2: 892 | Bate com a Gold 1? True
Registros: 5606 | Esperado 5606? True
Área vaga bate com a Gold 1? True


,CNPJ_Fundo_Classe,ano,Trimestre,periodo,qtd_registros,area_total_m2,area_vaga_m2,vacancia_ponderada
0,02.508.235/0001-14,2020,1,2020-T1,1,348156.92,50000.207909,0.143614
1,05.437.916/0001-27,2020,1,2020-T1,3,36566.47,5123.878337,0.140125
2,11.728.688/0001-47,2020,1,2020-T1,14,490255.40,31334.943778,0.063916
3,11.839.593/0001-09,2020,1,2020-T1,6,76413.69,0.000000,0.000000
4,12.887.506/0001-43,2020,1,2020-T1,4,23662.43,7165.600000,0.302826
5,12.978.943/0001-72,2020,1,2020-T1,3,63676.37,0.000000,0.000000
6,13.974.819/0001-00,2020,1,2020-T1,2,102891.38,0.000000,0.000000
7,15.098.422/0001-19,2020,1,2020-T1,1,41004.31,41004.310000,1.000000
8,16.671.412/0001-93,2020,1,2020-T1,3,182484.75,8296.628000,0.045465
9,17.143.998/0001-86,2020,1,2020-T1,1,55202.00,2036.953800,0.036900


## Conclusões do notebook 03

**Tabelas Gold geradas**

- `gold/vacancia_segmento_trimestral.parquet` (Gold 1): 26 trimestres, de 2020-T1 a 2026-T2, com 5.606 registros de imóvel.
- `gold/vacancia_fundo_trimestral.parquet` (Gold 2): 892 linhas (fundo × trimestre), com os mesmos 5.606 registros.

**Decisões tomadas**

- Agrupamento por ano + trimestre, mantendo a data mais recente de cada fundo no trimestre (DA-24). Único caso: fundo 13.012.312/0001-67 no 2023-T1, mesmo imóvel (Pilkington) informado em 31/01 e 31/03.
- Regra centralizada na view `imoveis_trimestre` (DA-25).
- Fundo identificado só por CNPJ (DA-26).
- Gold 3 não criada, porque o conteúdo já está na Gold 1 (DA-27).

**Validações**

- A view `imoveis_trimestre` tem 5.606 registros, igual à Gold 1.
- A Gold 2 bate com a Gold 1 em linhas (892 = soma de `qtd_fundos`), em registros e em área vaga total.

**Pontos para a EDA (notebook 04)**

- Até 2025-T3, a vacância ponderada fica abaixo da simples: a vacância se concentra em galpões pequenos.
- Em 2025-T4, a área vaga triplica (277 mil → 849 mil m²) com área total estável. Investigar quais fundos explicam o salto, usando a Gold 2.
- Em 2026, registros e área total caem (378 → 275 → 201 registros). É preciso verificar se o 2026-T2 está incompleto ou se a composição de fundos mudou.
- A amostra cresce de 35 fundos (2024-T4) para 48 (2025-T1). A mudança na composição pode afetar a comparação ao longo do tempo.